# EducAgent: concept & skill extraction (pass 1)

Runs the repo's extraction pipeline (`eduagent_kg.extraction.cli`) against an open-source model
served by **vLLM** on Kaggle's GPUs. Output: one validated graph fragment per lesson.

**Before running**
1. Settings → Accelerator: **GPU T4 x2**, Internet: **On**.
2. Add the dataset you made from `data/kaggle/eduagent-kg-input.zip` (Add Input → your datasets).
3. Run with `MAX_SECTIONS = 3` first, check the output, then set it to `None` for the whole book.

Re-running is safe: finished sections are cached in `/kaggle/working/extraction`.

In [ ]:
# ---- settings -------------------------------------------------------------------
MODEL = "Qwen/Qwen3-14B-AWQ"   # 4-bit AWQ, fits on 2x T4 with tensor parallelism
MAX_SECTIONS = 3               # None = whole book
CONCURRENCY = 4                # vLLM batches parallel requests
MAX_TOKENS = 6000              # per-section output budget (old run was cut off near ~1000)
PORT = 8000

In [ ]:
import glob, os, shutil, sys
matches = glob.glob("/kaggle/input/**/eduagent_kg/__init__.py", recursive=True)
assert matches, "Add the eduagent-kg-input dataset to this notebook first"
INPUT_ROOT = os.path.dirname(os.path.dirname(matches[0]))
SECTIONS = sorted(glob.glob(f"{INPUT_ROOT}/sections/*.jsonl"))[0]
SRC = "/kaggle/working/src"
shutil.rmtree(SRC, ignore_errors=True)
shutil.copytree(f"{INPUT_ROOT}/eduagent_kg", f"{SRC}/eduagent_kg")
sys.path.insert(0, SRC)
print("sections file:", SECTIONS)

In [ ]:
!pip install -q vllm pydantic-settings openai

In [ ]:
import subprocess, time, urllib.request
log = open("/kaggle/working/vllm.log", "w")
server = subprocess.Popen([
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL, "--served-model-name", "qwen3-14b",
    "--tensor-parallel-size", "2", "--dtype", "half",
    "--max-model-len", "16384", "--gpu-memory-utilization", "0.90",
    "--port", str(PORT),
], stdout=log, stderr=subprocess.STDOUT)

deadline = time.time() + 20 * 60   # first start downloads ~10 GB of weights
while time.time() < deadline:
    if server.poll() is not None:
        raise RuntimeError("vLLM exited:\n" + open("/kaggle/working/vllm.log").read()[-3000:])
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/v1/models", timeout=5)
        print("vLLM is ready"); break
    except Exception:
        time.sleep(10)
else:
    raise TimeoutError(open("/kaggle/working/vllm.log").read()[-3000:])

In [ ]:
import subprocess
cmd = [sys.executable, "-m", "eduagent_kg.extraction.cli", SECTIONS,
       "--output", "/kaggle/working/extraction",
       "--base-url", f"http://localhost:{PORT}/v1", "--model", "qwen3-14b",
       "--max-tokens", str(MAX_TOKENS), "--concurrency", str(CONCURRENCY)]
if MAX_SECTIONS:
    cmd += ["--max-sections", str(MAX_SECTIONS)]
result = subprocess.run(cmd, cwd=SRC, env={**os.environ, "PYTHONPATH": SRC})
print("exit code", result.returncode)

In [ ]:
# Quick look at the result
import json, pathlib
root = next(pathlib.Path("/kaggle/working/extraction").glob("*/fragments"))
for path in sorted(root.glob("*.json")):
    frag = json.loads(path.read_text())
    nodes = [e for e in frag["entities"] if e["node_type"] in ("Concept", "Skill")]
    print(f"\n{frag['section_id']}: {len(nodes)} nodes, {len(frag['warnings'])} dropped")
    for e in nodes:
        print(f"  [{e['node_type']}] {e['name']} - {e.get('description') or ''}")

In [ ]:
# Merge fragments into one graph and score it (same report as on your PC)
import pathlib
from eduagent_kg.evaluation.quality import evaluate_graph, render_markdown
from eduagent_kg.resolution.merger import HierarchicalGraphMerger, load_fragments
from eduagent_kg.schemas.documents import StructuredTextbook

out = pathlib.Path("/kaggle/working/extraction")
textbook = StructuredTextbook.model_validate_json((out / "textbook.json").read_text(encoding="utf-8"))
fragments = load_fragments(next(out.glob("*/fragments")))
graph, queue, stats = HierarchicalGraphMerger().merge(fragments, textbook)
(out / "canonical-graph.json").write_text(graph.model_dump_json(indent=2), encoding="utf-8")
(out / "review-queue.json").write_text(queue.model_dump_json(indent=2), encoding="utf-8")
print(render_markdown({"new": evaluate_graph(graph)}))


In [ ]:
# Step 3: merge duplicate concepts/skills with sentence embeddings (BAAI/bge-small-en-v1.5, on CPU).
result = subprocess.run([sys.executable, "-m", "eduagent_kg.resolution.embedding_dedup", str(out)],
                        cwd=SRC, env={**os.environ, "PYTHONPATH": SRC})
print("exit code", result.returncode)

In [ ]:
# Step 4: book-level prerequisites across lessons and chapters (needs the vLLM server from above).
# One call per lesson; answers are cached in extraction/prerequisites-raw, so re-running is free.
result = subprocess.run([sys.executable, "-m", "eduagent_kg.resolution.book_prerequisites", str(out),
                         "--base-url", f"http://localhost:{PORT}/v1", "--model", "qwen3-14b",
                         "--concurrency", str(CONCURRENCY)],
                        cwd=SRC, env={**os.environ, "PYTHONPATH": SRC})
print("exit code", result.returncode)
from eduagent_kg.schemas.graph import CanonicalGraph
final = CanonicalGraph.model_validate_json((out / "canonical-graph.prereq.json").read_text(encoding="utf-8"))
print(render_markdown({"merged": evaluate_graph(graph), "final": evaluate_graph(final)}))

In [ ]:
shutil.make_archive("/kaggle/working/extraction", "zip", "/kaggle/working/extraction")
print("Download /kaggle/working/extraction.zip from the Output panel")